# Scanpy PBMC quickstart

Audience:
- Single-cell researchers who already use Scanpy and want a model-focused baseline workflow.

Prerequisites:
- Install `scdlkit[tutorials]`.
- Familiarity with `AnnData` and basic Scanpy concepts.

Learning goals:
- Load PBMC data with Scanpy.
- Train a VAE baseline with `TaskRunner`.
- Store latent embeddings in `adata.obsm`.
- Continue with Scanpy neighbors and UMAP on the learned representation.

Install:
```bash
python -m pip install "scdlkit[tutorials]"
```

Links:
- Docs: <https://uddamvathanak.github.io/scDLKit/>
- Repository: <https://github.com/uddamvathanak/scDLKit>


## Outline

1. Load PBMC data with Scanpy.
2. Inspect the dataset and choose the label field.
3. Detect the runtime device.
4. Train a VAE with `device="auto"`.
5. Evaluate metrics and save artifacts.
6. Push the latent embedding into `adata.obsm`.
7. Run Scanpy neighbors and UMAP on the latent space.


In [ ]:
from __future__ import annotations

from pathlib import Path

import scanpy as sc
import torch
from IPython.display import display

from scdlkit import TaskRunner

DATA_PATH = Path("examples/data/pbmc3k_processed.h5ad")
OUTPUT_DIR = Path("artifacts/pbmc_vae_quickstart")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_name}")


## Load PBMC data

The tutorial prefers the repository copy of `pbmc3k_processed.h5ad` when available, then falls back to `scanpy.datasets.pbmc3k_processed()`.


In [ ]:
adata = sc.read_h5ad(DATA_PATH) if DATA_PATH.exists() else sc.datasets.pbmc3k_processed()
print(adata)
print(f"Shape: {adata.shape}")
print("obs columns:", list(adata.obs.columns))


## Train the VAE baseline

This is the main scDLKit step. The code is the same on CPU and GPU because the runner uses `device="auto"`.


In [ ]:
runner = TaskRunner(
    model="vae",
    task="representation",
    epochs=10,
    batch_size=128,
    label_key="louvain",
    device="auto",
    output_dir=str(OUTPUT_DIR),
)
runner.fit(adata)
metrics = runner.evaluate()
metrics


## Save a report and inspect the training curve

The quickstart writes a Markdown report and a training-loss plot to `artifacts/pbmc_vae_quickstart/`.


In [ ]:
runner.save_report(OUTPUT_DIR / "report.md")
loss_fig, _ = runner.plot_losses()
loss_fig.savefig(OUTPUT_DIR / "loss_curve.png", dpi=150, bbox_inches="tight")
display(loss_fig)


## Push the latent space back into Scanpy

scDLKit stays model-focused. Once the latent representation is available, continue the downstream neighborhood and visualization workflow with Scanpy.


In [ ]:
adata.obsm["X_scdlkit_vae"] = runner.encode(adata)
sc.pp.neighbors(adata, use_rep="X_scdlkit_vae")
sc.tl.umap(adata)
umap_fig = sc.pl.umap(adata, color="louvain", return_fig=True, frameon=False)
umap_fig.savefig(OUTPUT_DIR / "latent_umap.png", dpi=150, bbox_inches="tight")
umap_fig


## Expected outputs

After running this notebook you should have:

- metrics from `runner.evaluate()`
- `artifacts/pbmc_vae_quickstart/report.md`
- `artifacts/pbmc_vae_quickstart/loss_curve.png`
- `artifacts/pbmc_vae_quickstart/latent_umap.png`

Recommended next step:
- open the PBMC model-comparison tutorial to compare `autoencoder`, `vae`, and `transformer_ae` on the same dataset.
